# Bottom-Up 3D Wireframe Swarm Generation (Anchor + Edge Subdivision)

This notebook demonstrates the **Bottom-Up 3D Wireframe Swarm Generation Pipeline**:
- **Zero PyTorch / Point-E dependencies:** Pure geometric math running in $<1$ ms on CPU.
- **Topological Integrity:** Mandatory corner anchors guarantee shape recognizability even at ultra-low drone budgets ($N < 20$).
- **Proportional Edge Subdivision:** Remaining drone budget is distributed proportionally across structural edges according to Euclidean segment length.
- **Interactive 3D Visualizations:** Multi-scale comparisons and 4-shape gallery rendered with Plotly in dark mode.

## Cell 1: Imports & Loading `sample_wireframes.json`

Loads the geometric wireframe graph definitions and initializes Plotly for inline 3D rendering in VS Code.

In [ ]:
# ==============================================================================
# Cell 1: Imports & Loading sample_wireframes.json
# ==============================================================================
import json
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# Configure Plotly to render inline in VS Code and Jupyter notebooks
pio.renderers.default = "notebook+vscode"

def show_figure(fig: go.Figure):
    """
    Renders Plotly figures inline reliably in VS Code webview and Jupyter notebooks.
    Uses embedded HTML div to ensure 100% visibility without external browser popups.
    """
    try:
        html_code = fig.to_html(include_plotlyjs='cdn', full_html=False)
        display(HTML(html_code))
    except Exception:
        try:
            fig.show(renderer="notebook+vscode")
        except Exception:
            fig.show()

# Load wireframe graphs
with open("sample_wireframes.json", "r", encoding="utf-8") as f:
    wireframe_db = json.load(f)

print("=" * 68)
print("              LOADED WIREFRAME GEOMETRY DATABASE                      ")
print("=" * 68)
for name, data in wireframe_db.items():
    n_anchors = len(data["anchors"])
    n_edges = len(data["edges"])
    desc = data.get("description", "")
    print(f" • {name:<14} | Anchors (Corners): {n_anchors:2d} | Edges: {n_edges:2d} | {desc}")
print("=" * 68)

## Cell 2: Core Bottom-Up Subdivision Function (`build_swarm_from_graph`)

The algorithm allocates $N$ drones to a 3D wireframe graph:
1. **If $N \le K$ ($K$ anchors):** Uses Farthest Point Sampling (FPS) from the origin-farthest anchor to preserve the widest outer extremities.
2. **If $N > K$:**
   - Locks the first $K$ drones directly to all corner anchors (labeled `"anchor"`).
   - Computes Euclidean edge lengths $L_e = \|v_j - v_i\|_2$.
   - Proportionally allocates the remaining budget $(N - K)$ across edges based on segment length:
     $$\text{exact}_e = \frac{L_e}{\sum L} \cdot (N - K)$$
   - Distributes integer rounding remainders to edges with the largest fractional parts.
   - Generates evenly spaced interior drones along each edge:
     $$P_s = \left(1 - \frac{s}{m + 1}\right) v_i + \left(\frac{s}{m + 1}\right) v_j \quad \text{for } s = 1, \dots, m$$
3. **Helper:** [`compute_min_distance(points)`](#) calculates the minimum pairwise clearance distance $d_{\min}$.

In [ ]:
# ==============================================================================
# Cell 2: Core Bottom-Up Subdivision Function (build_swarm_from_graph)
# ==============================================================================
def build_swarm_from_graph(graph: dict, num_drones: int) -> tuple[np.ndarray, list[str]]:
    """
    Builds a 3D drone swarm formation from a wireframe graph for any target drone count N:
    - If num_drones <= K: FPS selects the widest outer extremities from anchors.
    - If num_drones > K: Locks all K anchors, then proportionally subdivides edges.
    
    Returns:
        points: np.ndarray of shape (num_drones, 3)
        point_types: list of strings ('anchor' or 'edge') of length num_drones
    """
    anchors = np.array(graph["anchors"], dtype=float)
    edges = graph["edges"]
    K = len(anchors)

    # Case 1: Ultra-small swarm (N <= K anchors) -> FPS selects widest anchors
    if num_drones <= K:
        selected_idx = np.zeros(num_drones, dtype=int)
        distances = np.full(K, np.inf)

        # Start with the point farthest from the origin
        selected_idx[0] = np.argmax(np.linalg.norm(anchors, axis=1))

        for i in range(1, num_drones):
            last_pt = anchors[selected_idx[i - 1]]
            dist_to_last = np.sum((anchors - last_pt) ** 2, axis=1)
            distances = np.minimum(distances, dist_to_last)
            selected_idx[i] = np.argmax(distances)

        points = anchors[selected_idx]
        point_types = ["anchor"] * num_drones
        return points, point_types

    # Case 2: Full swarm (N > K anchors) -> Lock all anchors + subdivide edges
    points = [pt for pt in anchors]
    point_types = ["anchor"] * K
    remaining = num_drones - K

    # Compute Euclidean edge lengths
    edge_lengths = np.array([
        np.linalg.norm(anchors[j] - anchors[i]) for i, j in edges
    ], dtype=float)
    total_len = np.sum(edge_lengths)

    # Proportional allocation
    exact = (edge_lengths / (total_len + 1e-8)) * remaining
    drones_per_edge = np.floor(exact).astype(int)
    remainder = remaining - np.sum(drones_per_edge)

    # Distribute integer rounding remainder to edges with highest fractional parts
    if remainder > 0:
        fractional = exact - drones_per_edge
        top_indices = np.argsort(fractional)[-remainder:]
        drones_per_edge[top_indices] += 1

    # Generate evenly spaced interior edge points
    for (i, j), m in zip(edges, drones_per_edge):
        if m > 0:
            for s in range(1, m + 1):
                t = s / (m + 1.0)
                pt = (1.0 - t) * anchors[i] + t * anchors[j]
                points.append(pt)
                point_types.append("edge")

    return np.array(points, dtype=float), point_types


def compute_min_distance(points: np.ndarray) -> float:
    """
    Computes the minimum pairwise Euclidean distance between any two drones:
    d_min = min_{a != b} ||p_a - p_b||_2
    """
    if len(points) < 2:
        return 0.0
    diffs = points[:, None, :] - points[None, :, :]
    dists = np.linalg.norm(diffs, axis=-1)
    np.fill_diagonal(dists, np.inf)
    return float(np.min(dists))


# Quick verification test across all shapes
print("Algorithm Verification:")
for shape_name, g in wireframe_db.items():
    pts, ptypes = build_swarm_from_graph(g, num_drones=30)
    dmin = compute_min_distance(pts)
    print(f" ✓ {shape_name:<14} -> N=30 | Anchors: {ptypes.count('anchor'):2d} | Edges: {ptypes.count('edge'):2d} | d_min: {dmin:.4f}")

## Cell 3: Multi-Scale Drone Count Comparison ($N = 12, 25, 45, 80$)

Renders a 1-row, 4-column interactive 3D Plotly visualization comparing `"fighter_jet"` at varying drone budgets:
- **Gold Markers (`#FFD700`, size=6):** Mandatory corner anchors.
- **Cyan Markers (`#00E5FF`, size=4):** Interpolated structural edge drones.
- **Faint Dashed Lines (`rgba(255, 255, 255, 0.25)`):** Underlying wireframe graph edges.
- **Aspect Ratio:** Locked to `cube` with dark-sky theme.

In [ ]:
# ==============================================================================
# Cell 3: Multi-Scale Drone Count Comparison (N = 12, 25, 45, 80)
# ==============================================================================
SHAPE_NAME = "fighter_jet"
SCALE_COUNTS = [12, 25, 45, 80]

graph = wireframe_db[SHAPE_NAME]
anchors = np.array(graph["anchors"], dtype=float)
edges = graph["edges"]

# Build underlying wireframe line segments (separated by None for high-perf single trace)
wire_x, wire_y, wire_z = [], [], []
for i, j in edges:
    wire_x.extend([anchors[i, 0], anchors[j, 0], None])
    wire_y.extend([anchors[i, 1], anchors[j, 1], None])
    wire_z.extend([anchors[i, 2], anchors[j, 2], None])

subplot_titles = []
for N in SCALE_COUNTS:
    pts, _ = build_swarm_from_graph(graph, N)
    d_min = compute_min_distance(pts)
    subplot_titles.append(f"<b>N = {N} Drones</b><br><span style='font-size:11px; color:#38bdf8;'>d_min = {d_min:.3f}</span>")

fig_scales = make_subplots(
    rows=1, cols=4,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=subplot_titles,
    horizontal_spacing=0.02
)

for col_idx, N in enumerate(SCALE_COUNTS, start=1):
    points, point_types = build_swarm_from_graph(graph, N)
    
    # 1. Background Wireframe Skeleton
    fig_scales.add_trace(
        go.Scatter3d(
            x=wire_x, y=wire_y, z=wire_z,
            mode="lines",
            line=dict(color="rgba(255, 255, 255, 0.22)", width=1.5, dash="dash"),
            hoverinfo="none",
            showlegend=(col_idx == 1),
            name="Wireframe Edge"
        ),
        row=1, col=col_idx
    )

    # 2. Anchor Drones (Gold)
    anchor_mask = [pt_type == "anchor" for pt_type in point_types]
    if any(anchor_mask):
        anchor_pts = points[anchor_mask]
        fig_scales.add_trace(
            go.Scatter3d(
                x=anchor_pts[:, 0], y=anchor_pts[:, 1], z=anchor_pts[:, 2],
                mode="markers",
                marker=dict(size=6, color="#FFD700", opacity=1.0, line=dict(color="#FFFFFF", width=0.5)),
                hovertext=[f"Anchor #{i+1}" for i in range(len(anchor_pts))],
                hoverinfo="text",
                showlegend=(col_idx == 1),
                name="Anchor Drones (Corners)"
            ),
            row=1, col=col_idx
        )

    # 3. Edge Drones (Cyan)
    edge_mask = [pt_type == "edge" for pt_type in point_types]
    if any(edge_mask):
        edge_pts = points[edge_mask]
        fig_scales.add_trace(
            go.Scatter3d(
                x=edge_pts[:, 0], y=edge_pts[:, 1], z=edge_pts[:, 2],
                mode="markers",
                marker=dict(size=4, color="#00E5FF", opacity=0.95),
                hovertext=[f"Edge Drone #{i+1}" for i in range(len(edge_pts))],
                hoverinfo="text",
                showlegend=(col_idx == 1),
                name="Subdivided Edge Drones"
            ),
            row=1, col=col_idx
        )

# Unified Scene Layout: Locked Cube Aspect Ratio
scene_cube = dict(
    aspectmode="cube",
    xaxis=dict(title="X", range=[-1.1, 1.1], backgroundcolor="#0b0f19", gridcolor="#1e293b", showbackground=True),
    yaxis=dict(title="Y", range=[-1.1, 1.1], backgroundcolor="#0b0f19", gridcolor="#1e293b", showbackground=True),
    zaxis=dict(title="Z", range=[-1.1, 1.1], backgroundcolor="#0b0f19", gridcolor="#1e293b", showbackground=True),
    camera=dict(eye=dict(x=1.35, y=1.35, z=1.15))
)

fig_scales.update_layout(
    template="plotly_dark",
    title=dict(
        text=(
            f"<b>Bottom-Up Multi-Scale Wireframe Swarm: '{SHAPE_NAME}'</b><br>"
            f"<span style='font-size:13px; color:#94a3b8;'>"
            f"Anchor corners (Gold) are preserved across all scales; edge resolution scales smoothly with N."
            f"</span>"
        ),
        x=0.5, xanchor="center"
    ),
    scene=scene_cube,
    scene2=scene_cube,
    scene3=scene_cube,
    scene4=scene_cube,
    margin=dict(l=10, r=10, b=20, t=90),
    legend=dict(orientation="h", yanchor="bottom", y=-0.08, xanchor="center", x=0.5)
)

show_figure(fig_scales)

## Cell 4: All Shapes Gallery (`fighter_jet`, `star_3d`, `chair`, `house_3d`)

Renders a $2 \times 2$ 3D Plotly subplot grid displaying all 4 shapes at a uniform drone count $N = 35$:
- **Top-Left:** `fighter_jet` (Swept wings with twin vertical tail fins)
- **Top-Right:** `star_3d` (5-pointed volumetric star with apex hubs)
- **Bottom-Left:** `chair` (4 legs, seat frame, tall backrest)
- **Bottom-Right:** `house_3d` (Gable roof and rectangular walls)

In [ ]:
# ==============================================================================
# Cell 4: All Shapes Gallery (fighter_jet, star_3d, chair, house_3d)
# ==============================================================================
NUM_DRONES_GALLERY = 35
all_shapes = list(wireframe_db.keys())

# Compute subplot titles with min distance
titles_2x2 = []
for name in all_shapes:
    g = wireframe_db[name]
    pts, _ = build_swarm_from_graph(g, NUM_DRONES_GALLERY)
    d_min = compute_min_distance(pts)
    titles_2x2.append(f"<b>{name.upper()}</b> (N = {NUM_DRONES_GALLERY}, d_min = {d_min:.3f})")

fig_gallery = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}],
           [{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=titles_2x2,
    horizontal_spacing=0.04,
    vertical_spacing=0.08
)

grid_positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (row, col), shape_name in zip(grid_positions, all_shapes):
    g = wireframe_db[shape_name]
    anchors = np.array(g["anchors"], dtype=float)
    edges = g["edges"]

    # 1. Wireframe Skeleton
    wx, wy, wz = [], [], []
    for i, j in edges:
        wx.extend([anchors[i, 0], anchors[j, 0], None])
        wy.extend([anchors[i, 1], anchors[j, 1], None])
        wz.extend([anchors[i, 2], anchors[j, 2], None])

    fig_gallery.add_trace(
        go.Scatter3d(
            x=wx, y=wy, z=wz,
            mode="lines",
            line=dict(color="rgba(255, 255, 255, 0.22)", width=1.5, dash="dash"),
            hoverinfo="none",
            showlegend=((row, col) == (1, 1)),
            name="Wireframe Skeleton"
        ),
        row=row, col=col
    )

    # 2. Swarm Drones
    points, point_types = build_swarm_from_graph(g, NUM_DRONES_GALLERY)

    # Anchor points (Gold)
    anchor_mask = [t == "anchor" for t in point_types]
    if any(anchor_mask):
        apts = points[anchor_mask]
        fig_gallery.add_trace(
            go.Scatter3d(
                x=apts[:, 0], y=apts[:, 1], z=apts[:, 2],
                mode="markers",
                marker=dict(size=6, color="#FFD700", opacity=1.0, line=dict(color="#FFFFFF", width=0.5)),
                hovertext=[f"Anchor #{i+1}" for i in range(len(apts))],
                hoverinfo="text",
                showlegend=((row, col) == (1, 1)),
                name="Anchor Drones (Corners)"
            ),
            row=row, col=col
        )

    # Edge points (Cyan)
    edge_mask = [t == "edge" for t in point_types]
    if any(edge_mask):
        epts = points[edge_mask]
        fig_gallery.add_trace(
            go.Scatter3d(
                x=epts[:, 0], y=epts[:, 1], z=epts[:, 2],
                mode="markers",
                marker=dict(size=4.5, color="#00E5FF", opacity=0.95),
                hovertext=[f"Edge Drone #{i+1}" for i in range(len(epts))],
                hoverinfo="text",
                showlegend=((row, col) == (1, 1)),
                name="Subdivided Edge Drones"
            ),
            row=row, col=col
        )

# Locked Aspect Ratio for all 4 scenes
scene_gallery_cube = dict(
    aspectmode="cube",
    xaxis=dict(title="X", range=[-1.1, 1.1], backgroundcolor="#0b0f19", gridcolor="#1e293b", showbackground=True),
    yaxis=dict(title="Y", range=[-1.1, 1.1], backgroundcolor="#0b0f19", gridcolor="#1e293b", showbackground=True),
    zaxis=dict(title="Z", range=[-1.1, 1.1], backgroundcolor="#0b0f19", gridcolor="#1e293b", showbackground=True),
    camera=dict(eye=dict(x=1.35, y=1.35, z=1.15))
)

fig_gallery.update_layout(
    template="plotly_dark",
    title=dict(
        text=(
            f"<b>3D Wireframe Drone Swarm Gallery (N = {NUM_DRONES_GALLERY} Drones Each)</b><br>"
            f"<span style='font-size:13px; color:#94a3b8;'>"
            f"Rotatable 3D night-sky formations generated via Bottom-Up Anchor + Edge Subdivision"
            f"</span>"
        ),
        x=0.5, xanchor="center"
    ),
    height=800,
    scene=scene_gallery_cube,
    scene2=scene_gallery_cube,
    scene3=scene_gallery_cube,
    scene4=scene_gallery_cube,
    margin=dict(l=10, r=10, b=20, t=90),
    legend=dict(orientation="h", yanchor="bottom", y=-0.04, xanchor="center", x=0.5)
)

show_figure(fig_gallery)